In [2]:
"""
Machine Translation with Hugging Face: MarianMT and T5

Install:
    pip install transformers torch sentencepiece sacremoses
"""

import torch
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    T5ForConditionalGeneration,
    T5Tokenizer,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ---------------------------------------------------------------
# 1. MarianMT  (one model per language pair: Helsinki-NLP/opus-mt-{src}-{tgt})
# ---------------------------------------------------------------
def load_marian(src: str = "en", tgt: str = "fr"):
    model_name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(DEVICE)
    model.eval()
    return tokenizer, model


def translate_marian(texts, tokenizer, model, num_beams=5, max_length=128):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, num_beams=num_beams, max_length=max_length, early_stopping=True
        )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


# ---------------------------------------------------------------
# 2. T5  (pretrained for en->de, en->fr, en->ro via task prefix)
# ---------------------------------------------------------------
def load_t5(model_name: str = "t5-small"):
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(DEVICE)
    model.eval()
    return tokenizer, model


def translate_t5(texts, tokenizer, model, src="English", tgt="German",
                 num_beams=4, max_length=128):
    if isinstance(texts, str):
        texts = [texts]
    prompts = [f"translate {src} to {tgt}: {t}" for t in texts]
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_length
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, num_beams=num_beams, max_length=max_length, early_stopping=True
        )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


# ---------------------------------------------------------------
# Demo
# ---------------------------------------------------------------
if __name__ == "__main__":
    sentences = [
        "Machine translation is a fascinating field of research.",
        "How are you doing today?",
        "I would like to book a table for two.",
    ]

    # MarianMT: English -> French
    print("=== MarianMT (en -> fr) ===")
    m_tok, m_model = load_marian("en", "fr")
    for src, out in zip(sentences, translate_marian(sentences, m_tok, m_model)):
        print(f"{src}\n  -> {out}\n")

    # MarianMT: English -> Hindi
    print("=== MarianMT (en -> hi) ===")
    hi_tok, hi_model = load_marian("en", "hi")
    for src, out in zip(sentences, translate_marian(sentences, hi_tok, hi_model)):
        print(f"{src}\n  -> {out}\n")

    # T5: English -> German
    print("=== T5 (en -> de) ===")
    t_tok, t_model = load_t5("t5-small")
    for src, out in zip(sentences, translate_t5(sentences, t_tok, t_model, "English", "German")):
        print(f"{src}\n  -> {out}\n")

=== MarianMT (en -> fr) ===


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Machine translation is a fascinating field of research.
  -> Die Maschinenübersetzung ist ein faszinierendes Forschungsfeld.

How are you doing today?
  -> Wie tun Sie heute?

I would like to book a table for two.
  -> Ich möchte einen Tisch für zwei Personen buchen.

